In [38]:
import pandas as pd
import numpy as np
import fastparquet

In [39]:
####################################################################################################################
# Carrega Sinan
sinan = pd.read_parquet('/home/usuario/Documentos/Lucas/Projetos estudos/Escorpiao_pesa/Dados-processados/2_df_recodificado.parquet', 
                        filters=[('ANO', 'in', list(range(2015, 2020)))] # Com filtro interno
)

In [40]:
# Ajusta variaveis do SINAN

# Padroniza codigo municipio do Sinan como numero inteiro
sinan['ID_MN_RESI'] = sinan['ID_MN_RESI'].astype(int)

# Sinan para construir variavel MG
sinan_mg = sinan

# Converter ampolas para número
for col in ["NU_AMPOL_8", "NU_AMPOL_9"]:
    sinan_mg[col] = pd.to_numeric(sinan_mg[col], errors="coerce").fillna(0)

In [41]:

####################################################################################################################
# Cria variavel Moderado/Grave (MG)


# =========================================================
# FILTRO DE CASOS MODERADOS/GRAVES QUALIFICADOS
# ESCORPIONISMO
# =========================================================


# ---------------------------------------------------------
# CRITÉRIOS FORTES
# Isoladamente já sugerem fortemente MG
# ---------------------------------------------------------

criterio_forte = (

    # SAA >= 2 ampolas
    (sinan_mg["NU_AMPOL_8"] >= 2) |

    # SAEsc >= 2 ampolas
    (sinan_mg["NU_AMPOL_9"] >= 2) |

    # Óbito por animais peçonhentos
    (sinan_mg["EVOLUCAO"] == "Obito por ap") |

    # Manifestações vagais
    (sinan_mg["CLI_VAGAIS"] == "Sim")

)

# ---------------------------------------------------------
# CRITÉRIOS ASSOCIATIVOS
# Variáveis sujeitas a erro de preenchimento,
# mas que em conjunto aumentam a probabilidade
# de representar MG
# ---------------------------------------------------------

criterio_associativo = (

    # Soroterapia + classificação moderado/grave
    (
        (sinan_mg["CON_SOROTE"] == "Sim") &
        (sinan_mg["TRA_CLASSI"].isin(["Moderado", "Grave"]))
    ) |

    # Soroterapia + manifestações sistêmicas
    (
        (sinan_mg["CON_SOROTE"] == "Sim") &
        (sinan_mg["MCLI_SIST"] == "Sim")
    ) |

    # Soroterapia + complicações sistêmicas
    (
        (sinan_mg["CON_SOROTE"] == "Sim") &
        (sinan_mg["COM_SISTEM"] == "Sim")
    )

)

# ---------------------------------------------------------
# FILTRO FINAL
# ---------------------------------------------------------

filtro_mg = criterio_forte | criterio_associativo


# ---------------------------------------------------------
# CRIAR VARIÁVEL BINÁRIA
# ---------------------------------------------------------

sinan_mg["MG"] = np.where(filtro_mg, 1, 0)

# ---------------------------------------------------------
# CONFERÊNCIA
# ---------------------------------------------------------

print("Total de casos:", len(sinan_mg))
print("Moderados/Graves qualificados:", filtro_mg.sum())
print("Proporção:", round(filtro_mg.mean() * 100, 2), "%")


Total de casos: 117357
Moderados/Graves qualificados: 4475
Proporção: 3.81 %


In [60]:
# CRIAR COLUNAS DE TOTAIS

# TOTAIS GERAIS POR IDADE -------------------------------------------------------------------------------------
# Total de casos (sem distincao por faixa etaria)
total_casos = sinan_mg['ID_MN_RESI'].value_counts().reset_index(name='TOTAL_CASOS')

# Total ate 10 anos
total_10a = sinan_mg.loc[sinan_mg['IDADE_ANOS'] <= 10, 'ID_MN_RESI'].value_counts().reset_index(name='TOTAL_10A')

# Total ate 12 anos
total_12a = sinan_mg.loc[sinan_mg['IDADE_ANOS'] <= 12, 'ID_MN_RESI'].value_counts().reset_index(name='TOTAL_12A')

# Total ate 11 a 59 anos
total_11a59 = sinan_mg.loc[sinan_mg['IDADE_ANOS'].isin(range(11,59)), 'ID_MN_RESI'].value_counts().reset_index(name='TOTAL_12A')

# Total 60+
total_60a = sinan_mg.loc[sinan_mg['IDADE_ANOS'] >59, 'ID_MN_RESI'].value_counts().reset_index(name='TOTAL_12A')
# -------------------------------------------------------------------------------------------------------------

# TOTAIS DE MODERADOS/GRAVES POR IDADE ------------------------------------------------------------------------
# MG (total geral por municipio)
total_mg = sinan_mg.pivot_table(
    index='ID_MN_RESI', # linhas
    #values='ID_MN_RESI', # valores que serao contados
    columns='MG', # colunas
    fill_value=0, # valores ausentes serão preenchidos com zero
    aggfunc='size' # função de contagem
).reset_index()

total_mg.columns = ['ID_MN_RESI','LEVE','MG'] # Renomeia as colunas

# MG ate 10
mg_10 = sinan_mg[sinan_mg['IDADE_ANOS']<=10].pivot_table(
    index='ID_MN_RESI', # linhas
    #values='ID_MN_RESI', # valores que serao contados
    columns='MG', # colunas
    fill_value=0, # valores ausentes serão preenchidos com zero
    aggfunc='size' # função de contagem
).reset_index()

mg_10.columns = ['ID_MN_RESI','LEVE_10','MG_10'] # Renomeia as colunas


# MG ate 12
mg_12 = sinan_mg[sinan_mg['IDADE_ANOS']<=12].pivot_table(
    index='ID_MN_RESI', # linhas
    #values='ID_MN_RESI', # valores que serao contados
    columns='MG', # colunas
    fill_value=0, # valores ausentes serão preenchidos com zero
    aggfunc='size' # função de contagem
).reset_index()

mg_12.columns = ['ID_MN_RESI','LEVE_12','MG_12'] # Renomeia as colunas

# MG 11 a 59
mg_11a59 = sinan_mg[sinan_mg['IDADE_ANOS'].isin(range(11,59))].pivot_table(
    index='ID_MN_RESI', # linhas
    #values='ID_MN_RESI', # valores que serao contados
    columns='MG', # colunas
    fill_value=0, # valores ausentes serão preenchidos com zero
    aggfunc='size' # função de contagem
).reset_index()

mg_11a59.columns = ['ID_MN_RESI','LEVE_11A59','MG_11A59'] # Renomeia as colunas


# MG 60+
mg_60 = sinan_mg[sinan_mg['IDADE_ANOS']>59].pivot_table(
    index='ID_MN_RESI', # linhas
    #values='ID_MN_RESI', # valores que serao contados
    columns='MG', # colunas
    fill_value=0, # valores ausentes serão preenchidos com zero
    aggfunc='size' # função de contagem
).reset_index()

mg_60.columns = ['ID_MN_RESI','LEVE_60','MG_60'] # Renomeia as colunas


## COLUNAS
colunas = [total_casos,total_10a,total_12a, total_11a59, total_60a, 
           total_mg, mg_10, mg_12, mg_11a59, mg_60]



## Carregar codigos IBGE

- Objetivo: Juntar com variaveis do SINAN, para depois juntar à base02.

In [58]:
ibge = pd.read_excel('Dados-apoio/proj-2015-2019-POP-GERAL-RIPSA.xlsx').iloc[:,[0,1]]


In [ ]:
for i in colunas:
    ibge.merge(
        how='left',
        left_on='IBGE',
        right=i,
        right_on='ID_MN_RESI',
        validate=True
    )

ValueError: "True" is not a valid argument. Valid arguments are:
- "1:1"
- "1:m"
- "m:1"
- "m:m"
- "one_to_one"
- "one_to_many"
- "many_to_one"
- "many_to_many"

In [27]:
#df.to_excel('Dados-iniciais/base02.xlsx')